# FactoryOS · Inventory of a large private TAR.GZ archive (Issue #437)

**Target:** `download_uberupc1_1701111933_77520.tar.gz` (3.35 GB) on the owner's Google Drive.

This notebook uses **the owner's own Google Colab runtime** to download the private archive, without making it public, and produces a safe file-name-only inventory. It does **not** extract, execute, publish, or modify sites. The archive is stored temporarily in Colab's VM and may be erased when the session ends.

1. Run the cell, choose the Google account with access to the archive, and approve Drive read permissions.
2. Leave the notebook running while it downloads ~3.35 GB. Colab free-tier space/quota are not guaranteed.
3. The cell prints a short summary and saves `factoryos_archive_inventory.json` locally.
4. **Download the small inventory JSON from Colab and attach it in our chat.** We will identify the site roots and move only the selected safe components into the existing Render sandbox. Do not upload raw secrets, backups, .env, or databases.

Strict cost: do not enable Colab paid runtime, GPU, or additional billing. Stop if the runtime asks for payment.


In [ ]:
# FactoryOS Sandbox · metadata-only analysis, without extraction or execution.
FILE_ID = "12ZpUnRdjqz5sn7gyXbvGC2tk1SFHKB49"
MAX_ARCHIVE_BYTES = 4_000_000_000

import collections
import json
import os
import shutil
import tarfile
import time
from pathlib import Path, PurePosixPath
from google.colab import auth, files
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth

auth.authenticate_user()
credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
meta = drive_api.files().get(fileId=FILE_ID, fields="id,name,mimeType,size,parents").execute()
size = int(meta["size"])
assert 0 < size <= MAX_ARCHIVE_BYTES, "Unexpected archive size; review before downloading"
assert meta["name"].lower().endswith(".tar.gz"), "This notebook supports TAR.GZ only"
free_bytes = shutil.disk_usage("/content").free
print("Archive:", meta["name"], "| bytes:", size, "| free local bytes:", free_bytes)
assert free_bytes >= size + 2_000_000_000, "Not enough free Colab disk; do not download"

target = Path("/content/factoryos_source_archive.tar.gz")
if not target.is_file() or target.stat().st_size != size:
    download = drive_api.files().get_media(fileId=FILE_ID)
    with target.open("wb") as stream:
        downloader = MediaIoBaseDownload(stream, download, chunksize=8*1024*1024)
        done = False
        progress_last = -1
        while not done:
            status, done = downloader.next_chunk(num_retries=3)
            percent = int(status.progress()*100) if status else 0
            if percent >= progress_last + 10:
                print("Downloaded", percent, "%")
                progress_last = percent
assert target.stat().st_size == size, "Incomplete archive"

print("Scanning TAR entries (names/sizes only; content is NOT extracted)...")
roots = collections.Counter()
nested = collections.Counter()
exts = collections.Counter()
markers = []
rejected = []
totals = collections.Counter()
samples = {}
candidate_names = {"index.html", "index.php", "wp-config.php", "package.json", "composer.json", "wp-content", "public_html"}
with tarfile.open(target, "r|gz") as archive:
    for member in archive:
        totals["entries"] += 1
        path = PurePosixPath(member.name)
        if path.is_absolute() or ".." in path.parts or "\x00" in member.name:
            if len(rejected) < 20:
                rejected.append(member.name[:200])
            totals["unsafe_paths"] += 1
            continue
        if not path.parts:
            continue
        roots[path.parts[0]] += 1
        if len(path.parts) > 1:
            nested["/".join(path.parts[:2])] += 1
        if member.isfile():
            totals["files"] += 1
            totals["bytes_uncompressed"] += member.size
            exts[path.suffix.lower() or "(none)"] += 1
        if member.issym() or member.islnk():
            totals["links"] += 1
        if path.name.lower() in candidate_names and len(markers) < 200:
            markers.append({"path":str(path)[:400], "size":member.size})
        if len(samples) < 40 and path.parts[0] not in samples:
            samples[path.parts[0]] = str(path)[:300]
        if totals["entries"] % 200_000 == 0:
            print("Listed", totals["entries"], "entries")

result = {
    "source_file_id": FILE_ID, "source_name": meta["name"],
    "compressed_bytes": size, "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "totals": dict(totals), "top_roots": roots.most_common(60),
    "top_root_pairs": nested.most_common(80),
    "file_extensions": exts.most_common(40),
    "site_markers": markers,
    "unsafe_paths_samples": rejected,
    "root_samples": samples,
}
output = Path("/content/factoryos_archive_inventory.json")
output.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({"totals": result["totals"],"top_roots":result["top_roots"][:15],"site_markers":markers[:30]},ensure_ascii=False,indent=2))
print("Inventory saved:", output, "|", output.stat().st_size, "bytes")
files.download(str(output))

